In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [1]:
!pip install fasttext

In [2]:
!pip install sentence_transformers

     ---------------------------------------- 0.0/138.0 kB ? eta -:--:--
     -- ------------------------------------- 10.2/138.0 kB ? eta -:--:--
     ---------- -------------------------- 41.0/138.0 kB 653.6 kB/s eta 0:00:01
     -------------------------------------- 138.0/138.0 kB 1.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/171.5 kB ? eta -:--:--
   ------------------------------------- - 163.8/171.5 kB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 171.5/171.5 kB 3.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.0 MB ? eta -:--:--
   - -------------------------------------- 0.2/9.0 MB 7.3 MB/s eta 0:00:02
   -- ------------------------------------- 0.6/9.0 MB 7.1 MB/s eta 0:00:02
   --- ------------------------------------ 0.9/9.0 MB 7.0 MB/s eta 0:00:02
   ----- ---------------------------------- 1.2/9.0 MB 7.2 MB/s eta 0:00:02
   ------- -------------------------------- 1.7/9.0 MB 7.6 MB/s eta 0:00:01
   --------- 

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [2]:
import pandas as pd
import numpy as np
import json
import fasttext
import re, os, shutil, json
from os import path
from numpy.linalg import norm
import pickle

# Stop Words Processing - for data cleaning
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
sw = set(stopwords.words('english'))
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shahe\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shahe\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
#drive.mount('/content/gdrive')

defaultPath = "C:\\Users\\shahe\\FYP-PROJECT\\project\\FYP-1\\FYP\\"
Dataset_Folder = defaultPath + "Dataset2\\"
Processed_Dataset_Pickle_Path = Dataset_Folder + "processed_dataset.pkl"
Relevant_Comments_List_Pickle_Path = Dataset_Folder + "relevant_comments_list.pkl"
FastTextModelPath = defaultPath + "FastTextModelFile/cc.en.300-001.bin"


Reading Dataset Files and Merging them

In [4]:
Dataset_Files = os.listdir(Dataset_Folder)

In [5]:
DFs = []
length = 0

for file in Dataset_Files:
  if file.endswith('.csv'):
    df = pd.read_csv(Dataset_Folder + file, usecols=["author", "text"])
    DFs.append(df)
    length = length + 1

In [6]:
# Merging DataFrames

dataset_df = pd.concat(DFs)
dataset_df.shape

(4006, 2)

In [7]:
dataset_df.head(5)

,author,text
0,matt7054,"""Hard""?, not really. It's just different, it's..."
1,davidleepayne64,And when you quickly start seeing and feeling ...
2,cebow6,I quit all processed foods and only eating cle...
3,itsjillintexas,💯agreed! 👏👏👏\n\nOne of the best quotes I ever ...
0,ameventco,I want to believe this. Though everything we’v...


Dataset Preprocessing

In [8]:
# Defining Text Preprocing Function

def remove_stop_words(text):
  text_tokens = word_tokenize(text)
  tokens_without_sw = [word for word in text_tokens if not word in sw]
  return ' '.join(map(str, tokens_without_sw))


def preprocess(text):

    text = text.lower()
    text = text.replace("\\n", " ").replace("\n", " ")      # remove \n
    text = remove_stop_words(text)                          # remove stop words
    text = re.sub(r'\b\w{1,2}\b', ' ', text)                # remove words with 1 or 2 characters only
    text = re.sub(r'[^ a-zA-Z0-9]+','', text)                  # remove special characters / remove all except characters and spaces

    if text == "nan" or text == "none":
        text = " "
    text = re.sub(' +', ' ', text)

    return text.strip()

FastText - Features Generation

In [9]:
fastText_model = fasttext.load_model('cc_en_300-001-001.bin')

In [10]:
def fasttext_EmbeddingsGenerator(text):
  text_embeddings = fastText_model.get_word_vector(text)
  return text_embeddings #returns one feature vector.

Sentence Transformer - Features Generation

In [11]:
def sentence_transformer_EmbeddingsGenerator(text):
  text_embeddings = model.encode(text)
  return text_embeddings

TF-IDF - Features Generation

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline

#print("no. of docs, no. of unique corpus words) : ", tfidf_features_array.shape)
#returns two dimensional feature array against the entire corpus(dataset).
def TFIDF_EmbeddingsGenerator(corpus):
  tfidf_model = TfidfVectorizer(stop_words='english')
  tfidf_model = tfidf_model.fit(corpus) #provide dataset.
  tfidf_features = tfidf_model.transform(corpus) #returns feature vectors.
  tfidf_features_array = tfidf_features.toarray() #converts to array.
  tfidf_features_list = []
  for arr in tfidf_features_array:
    tfidf_features_list.append(arr) #appending feature arrays into a list.
  return (tfidf_features_list, tfidf_model)

#generating feature array against the query.
#This function Takes in dataset's feature array and the query.
def TFIDF_QueryFeatureGeneration(tfidf_model, query):
  tfidf_features = tfidf_model.transform([query])
  tfidf_features_array = tfidf_features.toarray()
  return tfidf_features_array[0]

**Creating feature vectors of our Dataset**

In [13]:
# Looping over dataset to generate feature vectors
processed_text = []
fasttext_features = []
ST_features = []
tfidf_features = []

for index, row in dataset_df.iterrows():

  processed = preprocess(row["text"]) #sending data for cleaning.
  processed_text.append(processed) #appending in the list.

  # FastText Features Generator
  text_features = fasttext_EmbeddingsGenerator(processed) #takes in one comment at a time and returns features.
  fasttext_features.append(text_features)

  #Sentence Transformer Features Generator
  text_features2 = sentence_transformer_EmbeddingsGenerator(processed)
  ST_features.append(text_features2)

# TFIDF Features Generator
(tfidf_features, tfidf_model) = TFIDF_EmbeddingsGenerator(processed_text) #passing the corpus. The func returns the trained model and extracted features.


dataset_df["processed_text"] = processed_text
dataset_df["fasttext_features"] = fasttext_features
dataset_df["tfidf_features"] = tfidf_features
dataset_df["SentenceTransformer_features"] = ST_features

dataset_df.to_pickle(Processed_Dataset_Pickle_Path, compression='infer', protocol=5, storage_options=None) #saving features data into pickle file.

**Reading Processed Dataset**

In [14]:
dataset_df = pd.read_pickle(Processed_Dataset_Pickle_Path) #reading the file.
dataset_df.head(5)

,author,text,processed_text,fasttext_features,tfidf_features,SentenceTransformer_features
0,matt7054,"""Hard""?, not really. It's just different, it's...",hard really different choosing discipline life...,"[0.005280149, -0.00013317022, -0.0038584971, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.021705475, 0.021106299, -0.008713445, 0.115..."
1,davidleepayne64,And when you quickly start seeing and feeling ...,quickly start seeing feeling results inspirati...,"[-0.000931342, -0.009943034, -0.015503479, 0.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.05385817, 0.08206615, 0.033782396, -0.0033..."
2,cebow6,I quit all processed foods and only eating cle...,quit processed foods eating clean walking incl...,"[-0.0010047528, -0.004678243, -0.010162719, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.013167868, 0.030945806, 0.04950701, 0.0762..."
3,itsjillintexas,💯agreed! 👏👏👏\n\nOne of the best quotes I ever ...,agreed one best quotes ever read losing weight...,"[-0.008416502, 0.0048308084, 0.0046579763, 0.0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.09364073, 0.01598587, 0.04879566, 0.115423..."
0,ameventco,I want to believe this. Though everything we’v...,want believe though everything told continue t...,"[-0.0013766099, -0.000108126726, -0.0028231647...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.000119488286, 0.038193494, 0.016595503, 0.0..."


In [15]:
print("fasttext", type(dataset_df["fasttext_features"]))
print("tfidf   ", type(dataset_df["tfidf_features"]))
print("SentenceTransformer   ", type(dataset_df["SentenceTransformer_features"]))

fasttext <class 'pandas.core.series.Series'>
tfidf    <class 'pandas.core.series.Series'>
SentenceTransformer    <class 'pandas.core.series.Series'>


**Defining function to extract relevant comments w.r.t query**

In [16]:
def cosine_similarity(A, B):

  # compute cosine similarity to find the similarity between two vectors.Higher the value, higher the similarity.
  cosine = np.dot(A,B)/(norm(A)*norm(B))
  return cosine


def get_relevant_comments_indices(query, comments, top_relevant_comments):
  #finding out relevant comments by returning indexes.
#comments: list of feature vectors.
  cosine_similarities = {}
  index = 0

  for comment in comments:
    cosine = cosine_similarity(query, comment) #calculating cosine similarity between query and the feature vector in feature vector list.
    cosine_similarities[cosine] = index #making a dictionary - key-value pair, cosine is the key and index is value.
    index = index + 1

  # Sort by key
  sorted_cosine_similarities = dict(sorted(cosine_similarities.items(), reverse = True)) #returning a dict of sorted keys(highest cosine value).
  print(sorted_cosine_similarities)
  # create a list of of relevant comments indices
  relevant_indices = []
  for i in range(0, top_relevant_comments): #top_relevant_comments= count.
    key, value = list(sorted_cosine_similarities.items())[i] #accessing top values to get the index of the comment.
    relevant_indices.append(value)

  return relevant_indices

**Ask user query and find relevant comments**

In [17]:
query = input()

# query text preprocessing
processed_query = preprocess(query)

# FastText Features Generator
#query_FastText_features = fasttext_EmbeddingsGenerator(processed_query)
# TFIDF Features Generator
#query_TFIDF_features = TFIDF_QueryFeatureGeneration(tfidf_model, processed_query)
# Sentence Transformer Generator
query_ST_features = sentence_transformer_EmbeddingsGenerator(processed_query)


indices = get_relevant_comments_indices(query_ST_features, dataset_df["SentenceTransformer_features"].tolist(), 10) #giving feature vectors of query and the dataset.

print(type(query_ST_features), type(dataset_df["SentenceTransformer_features"].tolist()))
print(type(query_ST_features[0]), type(dataset_df["SentenceTransformer_features"].tolist()[0]))

# I want to adopt keto diet

 i am 22 years old what diet suits me i work out too


{0.5195413: 3869, 0.5193182: 217, 0.51783216: 2032, 0.51738393: 1128, 0.51729465: 3912, 0.51011676: 114, 0.5080644: 1804, 0.5079007: 237, 0.5067574: 121, 0.4997182: 3360, 0.49524778: 83, 0.49495405: 3756, 0.4909825: 3769, 0.4863063: 1546, 0.48620474: 131, 0.48170102: 2438, 0.47976917: 2285, 0.4778319: 138, 0.47473913: 3720, 0.4732627: 3486, 0.47179475: 3829, 0.4711719: 68, 0.46921182: 1962, 0.4682066: 2127, 0.46607316: 1098, 0.46291476: 1257, 0.46269158: 323, 0.46221963: 3546, 0.4612317: 3527, 0.45984614: 3637, 0.4594201: 3690, 0.45928517: 1720, 0.45921773: 3597, 0.45767048: 3668, 0.45762545: 2148, 0.45584002: 2886, 0.45551088: 157, 0.45524842: 1784, 0.4535135: 1245, 0.45305344: 2336, 0.45284143: 3447, 0.45243907: 212, 0.4521314: 3122, 0.4516533: 3600, 0.45090514: 1483, 0.45037436: 175, 0.450235: 23, 0.45001733: 280, 0.4499946: 1480, 0.44982624: 573, 0.44957414: 2622, 0.4491128: 1757, 0.44896036: 3748, 0.448662: 3661, 0.44781893: 3646, 0.44778103: 352, 0.44751927: 3343, 0.44576308: 174

In [18]:
model_path = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
from transformers import pipeline
sentiment_task = pipeline("sentiment-analysis", model=model_path, tokenizer=model_path)
sentiment_task("Covid cases are increasing fast!")

C:\Users\shahe\Anaconda3\envs\torch\lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect 

[{'label': 'negative', 'score': 0.7235766649246216}]

In [19]:
arg = sentiment_task(query)

In [20]:
result = arg[0]

In [21]:
print("Label:", result['label'])

Label: neutral


In [22]:
indices

[3869, 217, 2032, 1128, 3912, 114, 1804, 237, 121, 3360]

In [23]:
relevancy_count = 1
relevant_comments_list = []
Comments_str = ""

print("Most Relevant Comments w.r.t Query\n")
for index in indices:
  comment = dataset_df.iloc[index]["text"]
  arg = sentiment_task(comment)
  result2 = arg[0]
  print(result2['label'])
  if(result2['label']== result['label'] or result['label']=='neutral'):
      print(str(relevancy_count) + " relevant comment: " + comment)
      relevancy_count = relevancy_count + 1
      relevant_comments_list.append(comment)
      Comments_str = Comments_str + comment + ". "


with open(Relevant_Comments_List_Pickle_Path, 'wb') as f:
  pickle.dump(relevant_comments_list, f)

Most Relevant Comments w.r.t Query

negative
1 relevant comment: I have 100 kg weight and my age is 15 years give me some diet plans and exersices
positive
2 relevant comment: ❤In 365 days lost 30kgs<br>❤Tshirt size XL to Medium<br>❤Feeling great.........i am living a quality life....Thank God.💥💥💥💥💥💥💥💥i am 35 yrs old but people treat me like a 25yr old🤣😅😂🤣😅😂thats the key and funpart of weight loss
neutral
3 relevant comment: I am a 68 year old female. I am 5 &#39;5 inches. I weight 144 pounds. I walk 3 miles a day. Only get sugar from fresh fruit, red meat once or twice a week. Half veggies on plate and I/4 protein, 1/4 carb. No desserts, no alcohol, no smoking, Water or coffee with just milk.  Doctors answer, well it&#39;s genetic take your meds : (    I feel like a failure. Oh I also never go out to eat. Only use olive oil on salads. Only whole wheat bread , when I splurge and eat a slice.
positive
4 relevant comment: 30 years old have a body of a 17 year old when I was in football f

In [24]:
print(relevant_comments_list)

['I have 100 kg weight and my age is 15 years give me some diet plans and exersices', '❤In 365 days lost 30kgs<br>❤Tshirt size XL to Medium<br>❤Feeling great.........i am living a quality life....Thank God.💥💥💥💥💥💥💥💥i am 35 yrs old but people treat me like a 25yr old🤣😅😂🤣😅😂thats the key and funpart of weight loss', 'I am a 68 year old female. I am 5 &#39;5 inches. I weight 144 pounds. I walk 3 miles a day. Only get sugar from fresh fruit, red meat once or twice a week. Half veggies on plate and I/4 protein, 1/4 carb. No desserts, no alcohol, no smoking, Water or coffee with just milk.  Doctors answer, well it&#39;s genetic take your meds : (    I feel like a failure. Oh I also never go out to eat. Only use olive oil on salads. Only whole wheat bread , when I splurge and eat a slice.', '30 years old have a body of a 17 year old when I was in football feel and look youthful still feeling the same been on a no sugar diet over a year it really pays off.', '50 years women is overweight want a 

**Text Sumarization**

In [25]:
!pip install evaluate

In [37]:
import nltk
import numpy as np
import pickle
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [25]:
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [46]:
from google.colab import drive
drive.mount('/content/gdrive')

defaultPath = "/content/gdrive/MyDrive/FYP1/"
Dataset_Folder = defaultPath + "Dataset/"
Processed_Dataset_Pickle_Path = Dataset_Folder + "processed_dataset.pkl"
Relevant_Comments_List_Pickle_Path = Dataset_Folder + "relevant_comments_list.pkl"
FastTextModelPath = defaultPath + "FastTextModelFile/cc.en.300-001.bin"


ModuleNotFoundError: No module named 'google.colab'

**Read relevant_comments_list**

In [26]:
relevant_comments_list = []

with open(Relevant_Comments_List_Pickle_Path, 'rb') as f:
  relevant_comments_list = pickle.load(f)

#relevant_comments_str = relevant_comments_list[0]
relevant_comments_str = ". ".join(relevant_comments_list) #merging all comments into a single string.


In [35]:
#model_name='SyedShaheer/bart-large-cnn-samsum_tuned_V2_1'
model_name = "SyedShaheer/my_awesome_billsum_model2_v2"
model2 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

In [36]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/295 [00:00<?, ?B/s]

In [32]:
dialogue = relevant_comments_str

prompt = f"""
Comment:

{dialogue}

Generate a summary?
"""

#generation_config = GenerationConfig(max_new_tokens=10, do_sample=True, temperature=0.5)

inputs = tokenizer(prompt, return_tensors='pt')


output = tokenizer.decode(
    model2.generate(
        inputs["input_ids"],
        max_new_tokens=100,
        #generation_config=generation_config,
        )[0],
    skip_special_tokens=True
)

print("-----\n")
print(f'INPUT PROMPT:\n{prompt}')

print("-----\n")
print(f'MODEL GENERATION - \n{output}\n')

-----

INPUT PROMPT:

Comment:

I have 100 kg weight and my age is 15 years give me some diet plans and exersices. ❤In 365 days lost 30kgs<br>❤Tshirt size XL to Medium<br>❤Feeling great.........i am living a quality life....Thank God.💥💥💥💥💥💥💥💥i am 35 yrs old but people treat me like a 25yr old🤣😅😂🤣😅😂thats the key and funpart of weight loss. I am a 68 year old female. I am 5 &#39;5 inches. I weight 144 pounds. I walk 3 miles a day. Only get sugar from fresh fruit, red meat once or twice a week. Half veggies on plate and I/4 protein, 1/4 carb. No desserts, no alcohol, no smoking, Water or coffee with just milk.  Doctors answer, well it&#39;s genetic take your meds : (    I feel like a failure. Oh I also never go out to eat. Only use olive oil on salads. Only whole wheat bread , when I splurge and eat a slice.. 30 years old have a body of a 17 year old when I was in football feel and look youthful still feeling the same been on a no sugar diet over a year it really pays off.. 50 years women

In [37]:
import numpy as np
from flask import Flask, request, render_template

app = Flask(__name__)

comments_and_answers = [
        
        ]
@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        input_value = request.form.get('input_value')

        # ----------------------------------------------------

        # TFIDF
        # query = input_value

        # # query text preprocessing
        # processed_query = preprocess(query)

        # # FastText Features Generator
        # query_FastText_features = fasttext_EmbeddingsGenerator(processed_query)
        # # TFIDF Features Generator
        # query_TFIDF_features = TFIDF_QueryFeatureGeneration(tfidf_model, processed_query)
        
        
        # indices = get_relevant_comments_indices(query_TFIDF_features, dataset_df["tfidf_features"].tolist(), 10) #giving feature vectors of query and the dataset.
        
        # print(type(query_TFIDF_features), type(dataset_df["tfidf_features"].tolist()))
        # print(type(query_TFIDF_features[0]), type(dataset_df["tfidf_features"].tolist()[0]))


        #--------------------------------------------------

        query = input_value
        
        # query text preprocessing
        processed_query = preprocess(query)
        
        # FastText Features Generator
        #query_FastText_features = fasttext_EmbeddingsGenerator(processed_query)
        # TFIDF Features Generator
        #query_TFIDF_features = TFIDF_QueryFeatureGeneration(tfidf_model, processed_query)
        # Sentence Transformer Generator
        query_ST_features = sentence_transformer_EmbeddingsGenerator(processed_query)
        
        
        indices = get_relevant_comments_indices(query_ST_features, dataset_df["SentenceTransformer_features"].tolist(), 10) #giving feature vectors of query and the dataset.
        
        print(type(query_ST_features), type(dataset_df["SentenceTransformer_features"].tolist()))
        print(type(query_ST_features[0]), type(dataset_df["SentenceTransformer_features"].tolist()[0]))
        
        # I want to adopt keto diet
        
        #------------------------------------------------------------

        arg = sentiment_task(query)
        result = arg[0]

        #------------------------------------------------------------
        
        # relevancy_count = 1
        # relevant_comments_list = []
        # Comments_str = ""
        
        # print("Most Relevant Comments w.r.t Query\n")
        # for index in indices:
        #   comment = dataset_df.iloc[index]["text"]
        #   print(str(relevancy_count) + " relevant comment: " + comment)
        #   relevancy_count = relevancy_count + 1
        #   relevant_comments_list.append(comment)
        #   Comments_str = Comments_str + comment + ". "
        
        
        # with open(Relevant_Comments_List_Pickle_Path, 'wb') as f:
        #   pickle.dump(relevant_comments_list, f)  


        #-------------------------------------------------------------

        relevancy_count = 1
        relevant_comments_list = []
        Comments_str = ""
        
        print("Most Relevant Comments w.r.t Query\n")
        for index in indices:
          comment = dataset_df.iloc[index]["text"]
          arg = sentiment_task(comment)
          result2 = arg[0]
          print(result2['label'])
          if(result2['label']== result['label'] or result['label']=='neutral'):
              print(str(relevancy_count) + " relevant comment: " + comment)
              relevancy_count = relevancy_count + 1
              relevant_comments_list.append(comment)
              Comments_str = Comments_str + comment + ". "
        
        
        with open(Relevant_Comments_List_Pickle_Path, 'wb') as f:
          pickle.dump(relevant_comments_list, f)

        #---------------------------------------------------------

        relevant_comments_list = []

        with open(Relevant_Comments_List_Pickle_Path, 'rb') as f:
          relevant_comments_list = pickle.load(f)

        #relevant_comments_str = relevant_comments_list[0]
        relevant_comments_str = ". ".join(relevant_comments_list) #merging all comments into a single string.
        #--------------------------------------------

        # model_name='SyedShaheer/distilbart-cnn-12-6_TUNED'
        # model2 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        # #----------------------------------------------

        # tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        #-----------------------------------------------

        dialogue = relevant_comments_str
        
        prompt = f"""
        Comment:
        
        {dialogue}
        
        Generate a summary?
        """
        
        #generation_config = GenerationConfig(max_new_tokens=10, do_sample=True, temperature=0.5)
        
        inputs = tokenizer(prompt, return_tensors='pt')
        
        
        output = tokenizer.decode(
            model2.generate(
                inputs["input_ids"],
                max_new_tokens=100,
                #generation_config=generation_config,
                )[0],
            skip_special_tokens=True
        )
        
        print("-----\n")
        print(f'INPUT PROMPT:\n{prompt}')
        
        print("-----\n")
        print(f'MODEL GENERATION - \n{output}\n')

        
        pred =  query
        ans= output   

        new_comment = {"question": query , "answer": output}
        
        comments_and_answers.append(new_comment)
        # Perform prediction using input_value here


        
        prediction = input_value  # Dummy prediction, replace with actual prediction
        return render_template('index.html', prediction=pred, answer=ans, comments_and_answers=comments_and_answers )
    return render_template('index.html', prediction=None)

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [15/May/2024 14:30:13] "GET / HTTP/1.1" 200 -


{0.8218853: 602, 0.7877576: 3578, 0.7701569: 3938, 0.7665896: 3927, 0.76121557: 3920, 0.756304: 312, 0.7484742: 3865, 0.7416311: 3760, 0.74041426: 509, 0.73436314: 147, 0.72426265: 3821, 0.7202334: 3880, 0.71678454: 3840, 0.7154149: 3751, 0.7130557: 451, 0.7119352: 3867, 0.7116881: 3971, 0.7089199: 3855, 0.7063969: 3733, 0.70178473: 3721, 0.69599104: 3595, 0.6929783: 3888, 0.69218063: 3566, 0.68946606: 3833, 0.6891098: 3613, 0.68856454: 3723, 0.6883923: 3508, 0.6867504: 238, 0.6859802: 3684, 0.68466604: 3859, 0.68287444: 3765, 0.6822119: 3914, 0.68060845: 3755, 0.67901784: 3795, 0.6769308: 1004, 0.67681897: 181, 0.6767155: 1074, 0.6760112: 3714, 0.6748235: 3594, 0.673703: 3842, 0.67347336: 3899, 0.6732573: 3606, 0.6719056: 4001, 0.6712522: 3740, 0.67045116: 3600, 0.66758746: 430, 0.6668779: 516, 0.6666468: 3592, 0.6665815: 3616, 0.6663107: 564, 0.6658935: 3788, 0.6653675: 3587, 0.6634614: 2937, 0.6630789: 1269, 0.66152877: 3802, 0.66137415: 1675, 0.6612989: 3929, 0.66087115: 1030, 0.66

127.0.0.1 - - [15/May/2024 14:30:44] "POST / HTTP/1.1" 200 -


-----

INPUT PROMPT:

        Comment:
        
        But Keto diet is high in fat and is advertised for weight loss !!!. losse skin After rapid weight lose (keto). after keto diet <br>what should I do for not gain weight. is it ok to follow your bodyweight exercise routine with keto diet?. Keto diet is effective for dramatic weight loss.. so, it&#39;s good for obese and overweight ppl, but make sure that it&#39;s for a short term only (no more than 4 months). I have a question about Keto. <br>I am now at a perfect weight. I do not want to lose more weight. How can I manage that being on Keto. <br>Looking forward to hearing from you. Thank you. Ursula. Now I&#39;m on diet for weight loss so how much time will I require to go in ketosis. Based on these pictures, the sum total is 1500-1700 calories.  Ketogenic or not, you&#39;ll lose weight on this.. Fasting and keto works , in fact it’s more effective than exercising for weight loss I lost 14 lbs without any exercises. I dont want to 

...

-----

INPUT PROMPT:

        Comment:
        
        Help I am an insomniac<br>When I try to sleep my all truma or bad experiences won&#39;t let me sleep. I have suffered from insomnia for most of my life and it is so difficult to function. I’ve tried everything! 9 hours of sleep would be heavenly!. I have been battling insomnia for over 10 years. I tried everything that was ever suggested but sleeping pills were the only thing that worked until I discovered edible canibus which is legal in my state. I can fall asleep with it, but I wake up 2 1/2 hours later and struggle with getting back to sleep and that can happen more than once a night. In the past I tried every imaginable remedy like over the counter sleeping aids as well as natural ones like Melotonin, Valerian root, kava, tryptophan, sleep teas, Magnesium drinks etc.  I went to a hypnotherapist, an acupuncturist. I listened to sleep music/stories.  Did yoga and meditation and breathing exercises.  Nothing worked consistently.

127.0.0.1 - - [15/May/2024 14:34:59] "POST / HTTP/1.1" 200 -


-----

INPUT PROMPT:

        Comment:
        
        When I stopped intermittent fasting, I gained a lot of weight. Amazing  lost 50 lbs,intermittent  fasting for 10 months,now on o.m.a.d.. thank  you DR.. Great info doc! Intermittent fasting offer several benefits like weight loss, improved metabolic health, and increased longevity. Thanks for sharing 🔥. Intermittent fasts is a norm in my life. If I fast for more then 36 hrs I feel sick and run down.
A higher metabolism makes the trip a short one.. I can exercise fasting no problem but after workout I feel super hungry. I was fasting with 2 meals a day and interval sprinting 2-3 times a week. I wasnt losing weight. I tried 1 meal a day. Dinner .and its working. I did intermittent fasting for 2 months...i lost around 12 kg..to then hit plateu..now i was wondering that if i stop fasting for a while..which for me its now been 3 months..so if I start fasting again i should start losing fat, but obviously at a slower rate....right?. So 

127.0.0.1 - - [15/May/2024 14:35:36] "POST / HTTP/1.1" 200 -


-----

INPUT PROMPT:

        Comment:
        
        I started the Paleo diet over two years ago.  I lost 25 lbs that I needed to loose within the first two months with no effort.  I do not eat dairy because I am lactose intolerant, so that was no hardship for me.  Getting rid of grains and legumes in my diet made a huge difference.  I had suffered from a variety of stomach ailments and they disappeared when I started this diet.  It is easy to follow, but the only drawback for some people is you have to be willing to cook.  I use no packaged or processed foods, so food prep is a little more labor intensive, but I&#39;ve found really good ways to do this.  I usually have a couple of days a week that I cook up soups, stews and casseroles and freeze them for later use.  However, I like to cook so this is no hardship for me.  Most of all, I have lots of energy, sleep better and feel good on this Paleo lifestyle.  I do &quot;cheat&quot; now and then and eat a white potato rather than a s

In [55]:
pred = "Your Question : " + query + "\nYour answer : " + output

In [56]:
print(pred)

Your Question : I want to adopt keto
Your answer : Ranveer credits Paaji's ketogenic diet for his successful weight loss journey. He credits his success to the support of Paaji and the Ahihek workoutzone. Paaji recommends the Agoge diet for weight loss but warns against the harmful effects of keto dieting. A nursing mother recommends Leviwiy Ketogenic Kaiserism as a potential alternative to keto for nursing mothers.
